In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

pools = {
    "yb_cbBTC": "0x83f24023d15d835a213df24fd309c47dAb5BEb32",
    "yb_wBTC": "0xD9FF8396554A0d18B2CFbeC53e1979b7ecCe8373",
    "yb_tBTC": "0xf1F435B05D255a5dBdE37333C0f61DA6F69c6127",
}
decimals = {
    "yb_cbBTC": 8,
    "yb_wBTC": 8,
    "yb_tBTC": 18,
}

In [ ]:
pool = pools["yb_wBTC"]
# pool = pools['yb_tBTC']
# pool = pools['yb_cbBTC']
# read pandas dataframe, all integers except header
pools_data = {}
donation_duration = 7 * 86_400
protection_period = 600
lp_threshold = 0.2
shares_max_ratio = 0.1
NORMALIZE = True
for pool, address in pools.items():
    filename = f"data/{address}.csv"
    data = pd.read_csv(filename)
    print(len(data))
    data = data[250::]
    ts = np.array(data["timestamp"]).astype(float)
    ts_dt = pd.to_datetime(ts, unit="s")
    blocks = np.array(data["block"]).astype(float)
    virtual_price = np.array(data["virtual_price"]).astype(float) / 1e18
    xcp_profit = np.array(data["xcp_profit"]).astype(float) / 1e18
    price_oracle = np.array(data["price_oracle"]).astype(float) / 1e18
    price_scale = np.array(data["price_scale"]).astype(float) / 1e18
    donation_shares = np.array(data["donation_shares"]).astype(float) / 1e18
    last_donation_release_ts = np.array(data["last_donation_release_ts"]).astype(float)
    donation_protection_expiry_ts = np.array(data["donation_protection_expiry_ts"]).astype(float)
    total_supply = np.array(data["totalSupply"]).astype(float) / 1e18
    D = np.array(data["D"]).astype(float) / 1e18
    balances_0 = np.array(data["balances_0"]).astype(float) / 1e18
    balances_1 = np.array(data["balances_1"]).astype(float) / 10 ** decimals[pool]
    # calc metrics
    xcp_profit_half = (xcp_profit - 1) / 2 + 1
    protection_factor = np.clip((donation_protection_expiry_ts - ts) / protection_period, 0, 1)
    t_elapsed = ts - last_donation_release_ts
    unlocked_shares = np.clip(donation_shares * t_elapsed / donation_duration, 0, donation_shares)
    available_shares = unlocked_shares * (1 - protection_factor)
    donations_proportion = donation_shares / total_supply
    unlocked_proportion = unlocked_shares / total_supply
    value_oracle = balances_0 + balances_1 * price_oracle
    value_pscale = balances_0 + balances_1 * price_scale
    vp_growth = virtual_price - virtual_price[0]
    xcp_profit_growth = xcp_profit - xcp_profit[0]
    xcp_profit_half_growth = xcp_profit_half - xcp_profit_half[0]
    pools_data[pool] = {
        "ts": ts,
        "ts_dt": ts_dt,
        "blocks": blocks,
        "virtual_price": virtual_price,
        "xcp_profit": xcp_profit,
        "price_oracle": price_oracle,
        "price_scale": price_scale,
        "donation_shares": donation_shares,
        "last_donation_release_ts": last_donation_release_ts,
        "donation_protection_expiry_ts": donation_protection_expiry_ts,
        "total_supply": total_supply,
        "D": D,
        "balances_0": balances_0,
        "balances_1": balances_1,
        "xcp_profit_half": xcp_profit_half,
        "protection_factor": protection_factor,
        "t_elapsed": t_elapsed,
        "unlocked_shares": unlocked_shares,
        "available_shares": available_shares,
        "donations_proportion": donations_proportion,
        "unlocked_proportion": unlocked_proportion,
        "value_oracle": value_oracle,
        "value_pscale": value_pscale,
        "vp_growth": vp_growth,
        "xcp_profit_growth": xcp_profit_growth,
        "xcp_profit_half_growth": xcp_profit_half_growth,
    }

## Price_oracle and price_scale

In [ ]:
pool_key = "yb_wBTC"
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["price_oracle"])
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["price_scale"])
# rotate x axis labels
plt.xticks(rotation=45)
# legend
plt.legend(["price_oracle", "price_scale"])
plt.show()

## virtual price and xcp_profit

In [ ]:
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["xcp_profit_growth"])
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["xcp_profit_half_growth"])
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["vp_growth"])
# rotate x axis labels
plt.xticks(rotation=45)

# legend
plt.legend(["xcp_profit", "xcp_profit/2", "virtual_price"])
plt.show()

## Donations

In [ ]:
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["protection_factor"])
plt.xticks(rotation=45)

# legend
plt.legend(["protection_factor"])
plt.show()

In [ ]:
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["donation_shares"])
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["unlocked_shares"])
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["available_shares"])
plt.xticks(rotation=45)
plt.legend(["donation_shares", "unlocked_shares", "available_shares"])
plt.show()

In [ ]:
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["donations_proportion"])
plt.plot(pools_data[pool_key]["ts_dt"], pools_data[pool_key]["unlocked_proportion"])
plt.xticks(rotation=45)
plt.legend(["donations_proportion", "unlocked_proportion"])
plt.show()

# Compare various pools

In [ ]:
def compare_pools_plots(pools, pools_data, metric):
    fig, ax = plt.subplots(figsize=(10, 4))
    for pool in pools:
        ax.plot(pools_data[pool]["ts_dt"], pools_data[pool][metric], label=pool)
    ax.tick_params(axis="x", rotation=45)
    ax.set_title(metric)
    ax.legend()
    fig.tight_layout()
    return fig, ax


def plot_metrics_panel(pools, pools_data, metrics):
    if not metrics:
        raise ValueError("metrics list must not be empty")

    fig, axes = plt.subplots(len(metrics), 1, figsize=(6, 3 * len(metrics)), sharex=True)
    if len(metrics) == 1:
        axes = [axes]

    for ax, metric in zip(axes, metrics):
        for pool in pools:
            ax.plot(pools_data[pool]["ts_dt"], pools_data[pool][metric], label=pool)
        ax.set_title(metric)
        ax.tick_params(axis="x", rotation=45)
        ax.grid(alpha=0.2)

    axes[-1].set_xlabel("timestamp")
    axes[0].legend(loc="upper left", bbox_to_anchor=(1.02, 1))
    fig.tight_layout()
    return fig, axes


metrics = [
    "virtual_price_growth",
    "xcp_profit_growth",
    "donation_shares",
    "available_shares",
    "price_scale",
    "price_oracle",
]
# metrics = ['virtual_price', 'balances_0', 'balances_1']
# metrics = ['donation_shares', 'available_shares', 'virtual_price']

# pools_filtered = {k: v for k,v in pools.items() if k in ['yb_wBTC','yb_cbBTC']}
pools_filtered = pools
fig, axes = plot_metrics_panel(pools_filtered, pools_data, metrics)
plt.show()

In [ ]:
import time
import datetime

idx_begin = 320
metric = "xcp_profit"
for pool in pools:
    idx_end = len(pools_data[pool][metric]) - 1
    # t_end = pools_data[pool]['ts'][idx_end]
    t_end = time.time()
    dt = 86_400  # seconds until t_end
    idx_end = np.where(pools_data[pool]["ts"] < t_end)[0][-1]
    for idx_begin in range(250, idx_end):
        t_begin = pools_data[pool]["ts"][idx_begin]
        if t_end - t_begin < dt:
            metric_begin = pools_data[pool][metric][idx_begin]
            metric_end = pools_data[pool][metric][idx_end]
            hrs_ago = (t_end - t_begin) / 3600
            annualized_growth_rate = (
                (metric_end / metric_begin) ** (365 * 86400 / (t_end - t_begin)) - 1
            ) * 100
            t_begin_utc = datetime.datetime.fromtimestamp(t_begin, datetime.timezone.utc).strftime(
                "%Y/%m/%d %H:%M"
            )
            t_end_utc = datetime.datetime.fromtimestamp(t_end, datetime.timezone.utc).strftime(
                "%Y/%m/%d %H:%M"
            )
            print(f"[{pool}] from {t_begin_utc} to {t_end_utc}")
            print(
                f"pre: {metric_begin:4.5f}, post: {metric_end:4.5f}, diff: {metric_end - metric_begin:4.5f}, annualized: {annualized_growth_rate:.2f}%"
            )
            print("")
            break